## Printers - Dream 2000

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import re
import time

In [2]:
## -- Options --
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

In [3]:
Printers = []
seen_links = set()

In [4]:
SEARCH = "printer"
BASE_URL = f"https://dream2000.com/en/search?q={SEARCH}&type=product"
MAX_PAGES = 100   # safety limit

driver = webdriver.Chrome(options=options)
driver.get(BASE_URL)

In [5]:
JS_EXTRACT = r"""const seen = new Set();
const out = [];
document.querySelectorAll('a[href*="/products/"]').forEach(a => {
  const href = a.href.split('?')[0];
  if (seen.has(href)) return;
  // climb to the smallest ancestor that looks like a full product card
  let card = a;
  for (let i = 0; i < 8 && card.parentElement; i++) {
    card = card.parentElement;
    const links = new Set([...card.querySelectorAll('a[href*="/products/"]')].map(x => x.href.split('?')[0]));
    if (links.size > 1) { card = card.firstElementChild === a ? a : card; break; }
    if (/(LE|EGP)/.test(card.innerText) && card.querySelector('img')) break;
  }
  seen.add(href);
  out.push({
    href: href,
    text: card.innerText || "",
    name: (a.getAttribute('aria-label') || a.getAttribute('title') || a.innerText || "").trim(),
    image: (card.querySelector('img') || {}).src || ""
  });
});
return out;"""

In [6]:
def parse_card(c):
    text = c["text"]
    lines = [l.strip() for l in text.splitlines() if l.strip()]

    # name: link text, else the longest line that isn't price/stock/button text
    name = c["name"]
    if not name:
        cand = [l for l in lines if not re.search(r"LE|EGP|stock|basket|option|pickup|unit price", l, re.I)]
        name = max(cand, key=len) if cand else ""

    # price: first number next to LE / EGP (current price comes before the old one)
    m = re.search(r"(?:LE|EGP)\s*([\d,]+(?:\.\d+)?)|([\d,]+(?:\.\d+)?)\s*(?:LE|EGP)", text)
    price = (m.group(1) or m.group(2)) if m else "Product Not Available"

    s = re.search(r"out of stock|low in stock|in stock", text, re.I)
    stock = s.group(0).title() if s else "Not Available"

    return {"Name": name, "Link": c["href"], "Image": c["image"], "Price": price, "Stock": stock}

In [7]:
## -- Scraping --
for page in range(1, MAX_PAGES + 1):
    driver.get(f"{BASE_URL}&page={page}")

    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, 'a[href*="/products/"]'))
        )
    except TimeoutException:
        print(f"Page {page}: no products found. Stopping.")
        break

    time.sleep(3)  # let prices / lazy images load
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)

    cards = driver.execute_script(JS_EXTRACT)
    page_products = 0

    for c in cards:
        if c["href"] in seen_links:
            continue
        seen_links.add(c["href"])
        Printers.append(parse_card(c))
        page_products += 1

    print(
        f"Page {page} finished successfully. "
        f"Products collected on this page: {page_products}. "
        f"Total products collected: {len(Printers)}"
    )

    if page_products == 0:   # page repeated / past the last page
        print("Scraping finished. No more new products.")
        break

driver.quit()

Page 1 finished successfully. Products collected on this page: 10. Total products collected: 10
Page 2 finished successfully. Products collected on this page: 10. Total products collected: 20
Page 3 finished successfully. Products collected on this page: 10. Total products collected: 30
Page 4 finished successfully. Products collected on this page: 10. Total products collected: 40
Page 5 finished successfully. Products collected on this page: 10. Total products collected: 50
Page 6 finished successfully. Products collected on this page: 10. Total products collected: 60
Page 7 finished successfully. Products collected on this page: 10. Total products collected: 70
Page 8 finished successfully. Products collected on this page: 10. Total products collected: 80
Page 9 finished successfully. Products collected on this page: 10. Total products collected: 90
Page 10 finished successfully. Products collected on this page: 9. Total products collected: 99
Page 11: no products found. Stopping.


In [8]:
df_Printers = pd.DataFrame(Printers)
df_Printers.head()

,Name,Link,Image,Price,Stock
0,HP LaserJet M111w Printer,https://dream2000.com/en/products/hp-laserjet-...,https://dream2000.com/cdn/shop/files/new_proje...,"7,499.00",Not Available
1,HP Smart Tank printer 581 3 in 1,https://dream2000.com/en/products/hp-smart-tan...,https://dream2000.com/cdn/shop/files/hp-smart-...,1,Not Available
2,HP Smart Tank 582 All-in-One Printer 4A8D6A,https://dream2000.com/en/products/hp-smart-tan...,https://dream2000.com/cdn/shop/files/new_proje...,"8,199.00",Not Available
3,Brother Printer laser L1632W,https://dream2000.com/en/products/brother-prin...,https://dream2000.com/cdn/shop/files/l1632w-6....,"9,799.00",Not Available
4,HP DeskJet Ink Advantage 4376 All-in-One Print...,https://dream2000.com/en/products/hp-deskjet-i...,https://dream2000.com/cdn/shop/files/111111111...,"4,899.00",Not Available


In [9]:
df_Printers.to_csv("dream2000_printers.csv", index=False, encoding="utf-8-sig")

In [10]:
df = pd.read_csv("dream2000_printers.csv", encoding="utf-8-sig")

In [11]:
df

,Name,Link,Image,Price,Stock
0,HP LaserJet M111w Printer,https://dream2000.com/en/products/hp-laserjet-...,https://dream2000.com/cdn/shop/files/new_proje...,"7,499.00",Not Available
1,HP Smart Tank printer 581 3 in 1,https://dream2000.com/en/products/hp-smart-tan...,https://dream2000.com/cdn/shop/files/hp-smart-...,1,Not Available
2,HP Smart Tank 582 All-in-One Printer 4A8D6A,https://dream2000.com/en/products/hp-smart-tan...,https://dream2000.com/cdn/shop/files/new_proje...,"8,199.00",Not Available
3,Brother Printer laser L1632W,https://dream2000.com/en/products/brother-prin...,https://dream2000.com/cdn/shop/files/l1632w-6....,"9,799.00",Not Available
4,HP DeskJet Ink Advantage 4376 All-in-One Print...,https://dream2000.com/en/products/hp-deskjet-i...,https://dream2000.com/cdn/shop/files/111111111...,"4,899.00",Not Available
...,...,...,...,...,...
94,Rapoo NK2600 Membrane Office Keyboard Wired - ...,https://dream2000.com/en/products/rapoo-nk2600...,https://dream2000.com/cdn/shop/files/555_c07b7...,420.00,Not Available
95,Redragon BM-2892 1200 DPI Wired Optical Mouse ...,https://dream2000.com/en/products/redragon-bm-...,https://dream2000.com/cdn/shop/files/AAAAAAAAA...,160.00,Not Available
96,Tank Super Cool Mini Printed Bottle 650 ml Foo...,https://dream2000.com/en/products/tank-super-c...,https://dream2000.com/cdn/shop/files/new_proje...,199.00,Not Available
97,Logitech MX Master 3S Bluetooth Edition Wirele...,https://dream2000.com/en/products/logitech-mx-...,https://dream2000.com/cdn/shop/files/mouuuu.jp...,"4,899.00",Not Available


In [1]:
import pandas as pd
df = pd.read_csv("dream2000_printers.csv", encoding="utf-8-sig")
df.to_parquet("dream2000_printers.parquet", index=False)